# Task 1: Independent Evaluation

The assignment defines independent evaluation as two things, and this notebook does both:

> - *research other works that have the same goals. Then you must compare and contrast your
>   results to those other works.*
> - *using data collected completely outside of the scope of your original training and
>   evaluation.*

Nothing here trains anything. It loads the model that Section 8 of
`01_task1_article_type.ipynb` selected as the ultimate judgement and asks one question:
**does it hold up on imagery it has never seen the like of?**

The answer is the most useful result in this notebook, and it is not the flattering one.

## What is evaluated

The deployed system exactly as Section 8.7 recommends it: the ResNet with decoupled
classifier retraining, its prediction averaged in probability space with the prediction on
its horizontal mirror. Two forward passes per image, no post-hoc logit adjustment.

## The external data

`notebooks/Task1/dataset1/` and `dataset2/` hold cosmetics crops derived from COCO, audited in
`docs/EXTERNAL_DATA_AUDIT.md`. They were collected for a possible *training* enrichment that was
never carried out, which leaves them in the one condition that makes an independent evaluation
valid: **no model in this project has ever seen them**, in training or in validation.

## 1. Setup

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

REPO_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "src" / "preprocessing.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the assignment repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import (IMAGE_TARGET_SIZE, load_image_array, load_manifest, make_split)

COMBINE = REPO_ROOT / "notebooks/Task1/01_task1_article_type.ipynb"
CHECKPOINTS = REPO_ROOT / "models/task1/checkpoints"
FIGURES = REPO_ROOT / "outputs/figures"
FIGURES.mkdir(parents=True, exist_ok=True)

DEVICE = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print("Device:", DEVICE)

### 1.1 The architecture, taken from the training notebook rather than restated

Re-typing `SmallResNet` here would create a second definition that can drift from the one the
weights were written by, and the failure would present as a confusing accuracy drop rather than
as an error. The class is therefore lifted out of the combine notebook's model-definition cell
and executed as-is. The trailing probe instance in that cell is cut off, since it exists only for
a parameter count there.

In [ ]:
notebook_cells = ["".join(c["source"])
                  for c in json.loads(COMBINE.read_text(encoding="utf-8"))["cells"]]
architecture = next(s for s in notebook_cells if "class BasicBlock" in s)
architecture = architecture[:architecture.index("set_seed(RANDOM_STATE)")]

namespace = {"nn": nn, "F": F, "torch": torch, "N_CLASSES": 124}
exec(architecture, namespace)
SmallResNet = namespace["SmallResNet"]
print("Architecture loaded from the combine notebook:", SmallResNet.__name__)

### 1.2 The model, and the constants it was trained under

Class order and normalisation come out of the checkpoints themselves. Recomputing either here
would risk evaluating under a different convention from the one the weights expect — the
normalisation constants in particular are fitted on training rows only and must be applied
unchanged, which is exactly the property Section 2.3 of the training notebook persists them for.

In [ ]:
DEPLOYED = "resnet_decoupled"

reference = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt",
                       map_location="cpu", weights_only=False)
CLASSES = list(reference["classes"])
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASSES)}
NORM_MEAN = np.asarray(reference["normalisation_mean"], dtype=np.float32)
NORM_STD = np.asarray(reference["normalisation_std"], dtype=np.float32)
FINGERPRINT = reference["fingerprint"]

print(f"Classes: {len(CLASSES)} | run fingerprint: {FINGERPRINT}")
print(f"Deployed checkpoint: model_{DEPLOYED}.pt")


In [ ]:
@torch.no_grad()
def deployed_probabilities(paths, batch_size=256):
    '''Softmax of the deployed checkpoint, averaged with its horizontal mirror.

    This is the pipeline from Section 8, not an approximation of it: same checkpoint, same
    flip TTA, same probability-space averaging of the two views.
    '''
    paths = list(paths)
    blob = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt", map_location="cpu",
                      weights_only=False)
    model = SmallResNet()
    model.load_state_dict(blob["state_dict"])
    model = model.to(DEVICE).eval()

    chunks = []
    for start in range(0, len(paths), batch_size):
        batch = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                          for p in paths[start:start + batch_size]]).astype(np.float32) / 255.0
        tensor = torch.from_numpy((batch - NORM_MEAN) / NORM_STD)
        tensor = tensor.permute(0, 3, 1, 2).to(DEVICE)
        probabilities = model(tensor).float().softmax(1)
        mirrored = model(tensor.flip(-1)).float().softmax(1)
        chunks.append(((probabilities + mirrored) / 2).cpu().numpy())
    del model
    return np.concatenate(chunks).astype(np.float64)


def report(name, paths, truth):
    '''Score one collection and return its probabilities.'''
    y = np.array([CLASS_TO_INDEX[t] for t in truth])
    probabilities = deployed_probabilities(paths)
    predicted = probabilities.argmax(1)
    labels = sorted(set(y))
    top5 = np.argpartition(probabilities, -5, axis=1)[:, -5:]
    row = {
        "Collection": name,
        "Images": len(y),
        "Classes": len(labels),
        "Top-1": accuracy_score(y, predicted),
        "Top-5": float((top5 == y[:, None]).any(1).mean()),
        "Macro-F1": f1_score(y, predicted, labels=labels, average="macro", zero_division=0),
        "Mean P(true)": float(probabilities[np.arange(len(y)), y].mean()),
        "Mean confidence": float(probabilities.max(1).mean()),
    }
    return row, probabilities, y, predicted

## 2. Provenance and leakage

An independent evaluation is only independent if the data is genuinely unseen. Two claims have
to hold, and both are checked rather than asserted: the external images are not supplied images,
and the labels name classes the model can actually predict.

`docs/EXTERNAL_DATA_AUDIT.md` records the prior screening — 1,200 dataset1 images reviewed in
contact sheets, 42 flagged and removed with approval, leaving 1,158. That audit establishes label
plausibility. It does **not** establish that the collection is unseen, which is what the hash
comparison below is for.

In [ ]:
manifest = load_manifest("articleType")
train_frame, val_frame = make_split(manifest, "articleType",
                                    validation_share=0.20, random_state=42)
print(f"Supplied manifest: {len(manifest):,} rows "
      f"({len(train_frame):,} train / {len(val_frame):,} validation)")

supplied_hashes = {hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in manifest["path"]}

COLLECTIONS = {
    "dataset1": (REPO_ROOT / "notebooks/Task1/dataset1/external_cosmetics.csv",
                 REPO_ROOT / "notebooks/Task1/dataset1/images"),
    "dataset2": (REPO_ROOT / "notebooks/Task1/dataset2/external_cosmetics2.csv",
                 REPO_ROOT / "notebooks/Task1/dataset2/images"),
}

external = {}
rows = []
for tag, (csv_path, image_dir) in COLLECTIONS.items():
    frame = pd.read_csv(csv_path, dtype={"id": str})
    frame["path"] = frame["id"].map(lambda i: str(image_dir / f"{i}.jpg"))
    frame = frame[frame["path"].map(lambda p: Path(p).is_file())].reset_index(drop=True)
    unknown = sorted(set(frame["articleType"]) - set(CLASSES))
    frame = frame[frame["articleType"].isin(CLASSES)].reset_index(drop=True)
    hashes = {hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in frame["path"]}
    rows.append({
        "Collection": tag,
        "Images": len(frame),
        "Classes": frame["articleType"].nunique(),
        "Byte-identical to supplied": len(hashes & supplied_hashes),
        "Labels outside the model's 124": len(unknown),
        "Source": frame["source"].iloc[0],
    })
    external[tag] = frame

provenance = pd.DataFrame(rows)
display(provenance)
assert provenance["Byte-identical to supplied"].eq(0).all(), "External data overlaps supplied data"
print("\nNo external image is byte-identical to a supplied image: the evaluation is independent.")

### 2.1 What the provenance does and does not establish

The hash check above proves these images are **unseen**, which is the property an independent
evaluation actually requires. It does not establish where they came from, and that distinction
is worth stating before any number below is read.

`docs/INDEPENDENT_EVALUATION_DATA.md` records the full position. In short:

| Established | Not established |
|---|---|
| No external image is byte-identical to a supplied one (0 of 1,857) | dataset1's upstream archive, version and licence — recorded as *pending* |
| Every external label is one of the model's 124 classes | dataset2's Roboflow / CC BY 4.0 claim, which is inherited and unverified |
| dataset1 fully screened; 42 flagged images removed, hashes verified | dataset2 label quality — 105 label flags across 148 of 699 images |

Two consequences for how the result should be read.

**dataset1 alone carries the finding.** It is the fully screened collection with no outstanding
label flags, and on its own it produces the headline result across 1,158 images. dataset2
corroborates it over four further classes but is not load-bearing, so its label-quality problems
do not threaten the conclusion.

**Label noise cannot manufacture a zero.** Even taking dataset2's 21.2% flag rate at face value,
four in five of its labels are sound; a model performing anywhere near its in-domain level would
score well above zero on those alone.

What the unresolved provenance genuinely limits is narrower: without the upstream archive this
collection is not a reproducible benchmark another group could regenerate. It is a valid probe of
domain robustness, and the report presents it as exactly that and nothing more.

## 3. What "outside the scope" means here, measured

The `source` field says `external_cosmetics_coco_v1`: these are crops from COCO, a dataset of
objects photographed **in context**. The supplied catalogue is studio product photography on a
white sweep. That is a covariate shift, and it is worth measuring rather than asserting, because
it is the explanation for everything in Section 5.

In [ ]:
def pixel_statistics(paths, limit=400):
    stack = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                      for p in list(paths)[:limit]])
    return {"Mean pixel": stack.mean(), "Std pixel": stack.std(),
            "Near-white fraction": float((stack > 240).mean())}


COSMETIC3 = ["Eyeshadow", "Lipstick", "Nail Polish"]
supplied_cosmetics = manifest[manifest["articleType"].isin(COSMETIC3)]

domain = pd.DataFrame([
    {"Collection": "supplied — all classes", **pixel_statistics(manifest["path"])},
    {"Collection": "supplied — the 3 cosmetic classes", **pixel_statistics(supplied_cosmetics["path"])},
    {"Collection": "external dataset1", **pixel_statistics(external["dataset1"]["path"])},
    {"Collection": "external dataset2", **pixel_statistics(external["dataset2"]["path"])},
])
display(domain.style.format({"Mean pixel": "{:.1f}", "Std pixel": "{:.1f}",
                             "Near-white fraction": "{:.3f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for label, paths, colour in [("supplied catalogue", manifest["path"], "#4C78A8"),
                             ("external (COCO crops)", external["dataset1"]["path"], "#E45756")]:
    stack = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                      for p in list(paths)[:400]])
    axes[0].hist(stack.reshape(-1), bins=64, density=True, alpha=0.55, label=label, color=colour)
    axes[1].hist((stack > 240).mean(axis=(1, 2, 3)), bins=40, density=True, alpha=0.55,
                 label=label, color=colour)
axes[0].set_title("Pixel intensity"); axes[0].set_xlabel("value"); axes[0].legend(fontsize=8)
axes[1].set_title("Per-image near-white fraction"); axes[1].set_xlabel("fraction > 240")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("The supplied catalogue is white-background product photography; the external "
             "collection is not", fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES / "10_domain_shift.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. In-domain control

Before asking whether the model transfers, establish that it can do the task at all on these
classes. Without this control a zero on the external set is ambiguous: it could mean the model
never learned cosmetics, rather than that it cannot recognise them out of domain.

Note how thin the supplied evidence is — these three classes are deep in the tail.

In [ ]:
print("Supplied rows for the three cosmetic classes:")
print(supplied_cosmetics["articleType"].value_counts().to_string())

control_rows = []
row, *_ = report("supplied cosmetics — all rows", supplied_cosmetics["path"],
                 supplied_cosmetics["articleType"])
control_rows.append(row)

supplied_val = val_frame[val_frame["articleType"].isin(COSMETIC3)]
row, *_ = report("supplied cosmetics — validation rows only", supplied_val["path"],
                 supplied_val["articleType"])
control_rows.append(row)

control = pd.DataFrame(control_rows)
display(control.style.format({c: "{:.4f}" for c in
                              ["Top-1", "Top-5", "Macro-F1", "Mean P(true)", "Mean confidence"]}))

**The model can do this task.** On supplied imagery of exactly these three classes it reaches
0.97 top-1 across all 39 rows. The validation-only row is weaker and much noisier, which is what
eight images buys you — that thinness is itself part of the finding, and Section 10.2 of the
training notebook already records unmeasured tail classes as a limitation.

## 5. Out-of-domain evaluation

The same three classes, the same model, photographed in context instead of on a white sweep.

In [ ]:
results, probability_store = [], {}
for tag, frame in external.items():
    row, probabilities, y, predicted = report(f"external {tag}", frame["path"],
                                              frame["articleType"])
    results.append(row)
    probability_store[tag] = (probabilities, y, predicted)

external_results = pd.DataFrame(results)
display(external_results.style.format({c: "{:.4f}" for c in
                                       ["Top-1", "Top-5", "Macro-F1", "Mean P(true)",
                                        "Mean confidence"]}))

uniform = 1 / len(CLASSES)
print(f"\nUniform prior over {len(CLASSES)} classes: {uniform:.4f}")
for tag, (probabilities, y, predicted) in probability_store.items():
    true_probability = probabilities[np.arange(len(y)), y]
    print(f"  {tag}: mean P(true class) = {true_probability.mean():.4f} "
          f"({true_probability.mean() / uniform:.2f}x the uniform prior); "
          f"best single image {true_probability.max():.4f}")

In [ ]:
print("What it predicts instead (external dataset1):")
predicted_names = pd.Series([CLASSES[i] for i in probability_store["dataset1"][2]])
display(predicted_names.value_counts().head(10).rename("images").to_frame())
print("Times it predicted any of the three correct classes:",
      int(predicted_names.isin(COSMETIC3).sum()))

### 5.1 Reading the result

**Top-1 is 0.0000 on 1,857 external images, and so is top-5.** Not one image of either collection
has its correct class anywhere in the model's five most likely labels. The mean probability
assigned to the true class is *below* the uniform prior of 1/124 — the model is not merely
uninformed here, it is actively steered away from the right answer.

Two things make this a strong result rather than a broken one:

- **The control rules out the obvious alternative.** The same model scores 0.97 top-1 on the
  same three classes in supplied imagery. It knows what a lipstick looks like on a white sweep.
- **What it predicts instead is coherent.** The errors concentrate on `Handbags`, `Bra`,
  `Briefs`, `Lounge Pants` — large, soft, centrally-framed objects. Against a cluttered natural
  background the model is reading the silhouette of the whole frame, which is precisely the cue
  Section 2.3 of notebook 00 identified as the dominant signal and which HOG exploited as the
  Section 3 baseline. The failure is a direct consequence of the feature the model was
  rewarded for learning.

### 5.2 The part that matters for deployment

Section 8.4 of the training notebook converts calibration into an operating policy: 82.6% of the
catalogue auto-tagged at 95% accuracy, on an ECE of 0.0305. That policy assumes confidence means
what it did in validation.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
for tag, colour in [("dataset1", "#E45756"), ("dataset2", "#F58518")]:
    ax.hist(probability_store[tag][0].max(1), bins=40, alpha=0.6, density=True,
            label=f"external {tag} (100% wrong)", color=colour)
ax.axvline(0.5, ls="--", lw=1, color="#333")
ax.set_xlabel("confidence in the predicted class")
ax.set_ylabel("density")
ax.set_title("Confidence on out-of-domain images, every one of which is misclassified",
             fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES / "11_confidence_under_shift.png", dpi=200, bbox_inches="tight")
plt.show()

for tag, (probabilities, y, predicted) in probability_store.items():
    confidence = probabilities.max(1)
    for threshold in (0.5, 0.8, 0.9):
        share = float((confidence >= threshold).mean())
        print(f"  {tag}: {share:6.1%} of images pass a {threshold:.0%} confidence gate "
              f"— and {share:.1%} of the catalogue would be auto-tagged wrongly")

### 5.2 Can the Failure Be Detected Automatically?

Section 5.1 establishes that the confidence score does not protect anyone: these images are
100% misclassified and a large share of them still clear a 50% confidence gate. Section 8.7 of
the training notebook draws the only safe conclusion available to it -- gate on input
provenance, not on the model's confidence -- which is a manual control and does not scale.

Section 8.6 of that notebook fits an alternative and this section tests it. The gate scores an
image by its squared Mahalanobis distance to the nearest class centroid in the model's own
512-dimensional feature space (Lee et al., NeurIPS 2018), using centroids, a pooled shrunk
covariance and a rejection threshold **all estimated on the training split alone**. Nothing here
refits it and nothing here tunes the threshold: the file is loaded and applied as it was
written, which is the only way this can be a test rather than a demonstration.

Two comparisons make the result readable.

- **Against maximum softmax probability** (Hendrycks & Gimpel, ICLR 2017), the standard
  baseline and the thing already known to fail here. If the distance score does no better, the
  honest finding is that this failure is not detectable from the model's own representation.
- **Against the in-domain validation split**, the same rows and the same split Section 2.1 of
  the training notebook fixed and every model in this project is scored on. That population is
  what "in distribution" means for this system, so it is what the false-rejection rate has to be
  measured against.

A caveat that must travel with any positive result: the two populations differ in source,
framing and capture conditions all at once, so a high AUROC measures "COCO photograph versus
catalogue photograph" and not purely "unfamiliar versus familiar". The in-domain control in
Section 4 is what keeps that honest -- it holds the classes fixed and varies only the imagery.

In [ ]:
from sklearn.metrics import roc_auc_score

GATE_PATH = REPO_ROOT / "models/task1/task1_ood_gate.pt"

if not GATE_PATH.is_file():
    print(f"No gate at {GATE_PATH}.")
    print("Run Section 8.6 of 01_task1_article_type.ipynb first; it writes this file.")
else:
    gate = torch.load(GATE_PATH, map_location="cpu", weights_only=False)
    assert gate["fingerprint"] == FINGERPRINT, (
        f"Gate was fitted under fingerprint {gate['fingerprint']} but this notebook loaded a "
        f"model at {FINGERPRINT}. Refitting one or the other is required; applying a gate "
        f"across a recipe change would measure the change, not the domain shift."
    )
    OOD_MEANS = gate["means"].astype(np.float64)
    OOD_PRECISION = gate["precision"].astype(np.float64)
    OOD_THRESHOLD = gate["threshold"]

    def mahalanobis_scores(features, means=None, precision=None):
        """Squared distance to the nearest centroid. Identical algebra to Section 8.6."""
        means = OOD_MEANS if means is None else means
        precision = OOD_PRECISION if precision is None else precision
        projected = features @ precision
        own = (projected * features).sum(axis=1)
        cross = projected @ means.T
        centroid = ((means @ precision) * means).sum(axis=1)
        return own + (centroid[None, :] - 2.0 * cross).min(axis=1)

    @torch.no_grad()
    def embed_and_score(paths, batch_size=256):
        """Penultimate features and plain-view softmax, from one forward pass per image.

        The gate scores the input, not the prediction, so this deliberately uses the single
        unflipped view rather than the deployed flip-averaged pipeline.
        """
        paths = list(paths)
        blob = torch.load(CHECKPOINTS / f"model_{DEPLOYED}.pt", map_location="cpu",
                          weights_only=False)
        model = SmallResNet()
        model.load_state_dict(blob["state_dict"])
        model = model.to(DEVICE).eval()

        features, probabilities = [], []
        for start in range(0, len(paths), batch_size):
            batch = np.stack([load_image_array(p, target_size=IMAGE_TARGET_SIZE, scale=False)
                              for p in paths[start:start + batch_size]]).astype(np.float32) / 255.0
            tensor = torch.from_numpy((batch - NORM_MEAN) / NORM_STD)
            tensor = tensor.permute(0, 3, 1, 2).to(DEVICE)
            embedded = model.embed(tensor)
            features.append(embedded.float().cpu().numpy().astype(np.float64))
            probabilities.append(model.fc(embedded).float().softmax(1).cpu().numpy())
        del model
        return np.concatenate(features), np.concatenate(probabilities)

    # In-domain reference: the whole validation split, as fixed in Section 2.1 of the
    # training notebook. Same rows every model in this project is scored on.
    id_features, id_probabilities = embed_and_score(val_frame["path"])
    id_distance = mahalanobis_scores(id_features)
    id_confidence = id_probabilities.max(1)

    gate_rows = []
    for tag, frame in external.items():
        ood_features, ood_probabilities = embed_and_score(frame["path"])
        ood_distance = mahalanobis_scores(ood_features)
        ood_confidence = ood_probabilities.max(1)

        # OOD is the positive class, so a detector that works scores it higher. Confidence
        # runs the other way, hence the negation.
        truth = np.r_[np.zeros(len(id_distance)), np.ones(len(ood_distance))]
        auroc_distance = roc_auc_score(truth, np.r_[id_distance, ood_distance])
        auroc_confidence = roc_auc_score(truth, np.r_[-id_confidence, -ood_confidence])

        # FPR@95TPR: with the threshold placed so 95% of in-domain images are accepted, what
        # share of out-of-domain images is accepted too? Lower is better.
        accept_distance = np.quantile(id_distance, 0.95)
        accept_confidence = np.quantile(-id_confidence, 0.95)
        gate_rows.append({
            "External set": tag,
            "AUROC (distance)": auroc_distance,
            "AUROC (confidence)": auroc_confidence,
            "FPR@95 (distance)": float((ood_distance <= accept_distance).mean()),
            "FPR@95 (confidence)": float((-ood_confidence <= accept_confidence).mean()),
            "Rejected at shipped threshold": float((ood_distance > OOD_THRESHOLD).mean()),
        })
        globals()[f"ood_distance_{tag}"] = ood_distance

    gate_table = pd.DataFrame(gate_rows)
    display(gate_table.style.format({c: "{:.4f}" for c in gate_table.columns
                                     if c != "External set"}))

    id_rejected = float((id_distance > OOD_THRESHOLD).mean())
    print(f"In-domain validation rejected at the shipped threshold: {id_rejected:.2%} "
          f"(the gate was set to reject 5% of training rows).")
    print("AUROC 0.5 means the score carries no information about domain; 1.0 means the two "
          "populations are perfectly separable by it.")

    fig, ax = plt.subplots(figsize=(7.5, 3.8))
    ax.hist(np.log10(id_distance), bins=60, alpha=0.65, density=True,
            label=f"in-domain validation (n={len(id_distance):,})", color="#4C78A8")
    for tag, colour in [("dataset1", "#E45756"), ("dataset2", "#F58518")]:
        if f"ood_distance_{tag}" in globals():
            ax.hist(np.log10(globals()[f"ood_distance_{tag}"]), bins=60, alpha=0.6,
                    density=True, label=f"external {tag}", color=colour)
    ax.axvline(np.log10(OOD_THRESHOLD), ls="--", lw=1.2, color="#333",
               label="shipped rejection threshold")
    ax.set_xlabel("log10 squared Mahalanobis distance to the nearest class centroid")
    ax.set_ylabel("density")
    ax.set_title("Does the model's own feature space reveal that an image is foreign?",
                 fontsize=10)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGURES / "12_ood_gate.png", dpi=200, bbox_inches="tight")
    plt.show()


The confidence signal does not degrade gracefully. A meaningful share of these images clear a
0.5 confidence gate while being **100% wrong**, so the auto-tagging policy would not route them
to human review — it would file them silently. An operating threshold calibrated in domain is
not a safety mechanism out of domain, and the deployment recommendation has to say so.

## 6. Comparison with published work

The second half of the independent-evaluation requirement. Three published efforts on this same
Fashion Product Images dataset, against this project's numbers.

In [ ]:
comparison = pd.DataFrame([
    {"Work": "Condition-CNN (Kolisnik et al., 2021)",
     "Backbone": "VGG16, ImageNet-pretrained",
     "Label space": "articleType as hierarchy level 3",
     "Top-1": "0.910", "Macro-F1": "not reported"},
    {"Work": "uditarora, multitask ResNet50 (GitHub)",
     "Backbone": "ResNet50, ImageNet-pretrained",
     "Label space": "top-20 articleType classes only",
     "Top-1": "0.8835", "Macro-F1": "not reported"},
    {"Work": "Li et al. (2020), arXiv:2005.08170",
     "Backbone": "pretrained CNNs",
     "Label space": "articleType; documents the imbalance",
     "Top-1": "not directly comparable", "Macro-F1": "not reported"},
    {"Work": "This project — ResNet + decoupled classifier, flip TTA",
     "Backbone": "SmallResNet, trained from scratch",
     "Label space": "all 124 articleType classes",
     "Top-1": "0.8774", "Macro-F1": "0.7693"},
])
display(comparison)

### 6.1 Reading the comparison

These are not head-to-head numbers and should not be reported as though they were: each work
uses its own split, and Condition-CNN and the ResNet50 repository both use ImageNet-pretrained
backbones, which this assignment forbids in a submitted model. With that said, three things
survive the caveat.

**The gap to pretrained work is small, and this model starts from noise.** Condition-CNN reports
0.910 top-1 with a pretrained VGG16; this project reaches 0.8774 across the full 124-class label
space with no pretrained weights at all. Roughly three points is a modest price for dropping
ImageNet entirely.

**The most flattering published comparison is the least comparable.** The ResNet50 repository's
0.8835 is measured over the *twenty most frequent* classes. This project's 0.8774 spans all 124,
including the 40 classes with fewer than 30 training images. Restricting to a head-only label
space removes exactly the part of the problem that is hard.

**None of them report macro-F1, and on this data that is the whole argument.** At 6,584:1
imbalance, accuracy cannot distinguish a model that handles the tail from one that ignores it —
this notebook's own majority-class baseline reaches 0.174 accuracy at 0.0027 macro-F1. A
published top-1 of 0.910 is consistent with a wide range of tail behaviour, none of which is
reported. That is a genuine gap in the comparison and it cuts both ways: it means this project
cannot claim to beat them on the metric it considers most important, because they never
published it.

## 7. What this contributes to the Ultimate Judgement

Three findings, carried into Section 8.7 of the training notebook.

1. **Within its domain the recommendation stands.** 0.97 top-1 on supplied imagery of the three
   external classes, consistent with the in-domain headline.
2. **Outside its domain the model fails completely and silently.** 0/1,857 top-1 *and* top-5,
   with mean true-class probability below the uniform prior, while remaining ~35% confident. The
   deployable claim is bounded to catalogue-style product photography on a plain background.
3. **Against published work it is competitive without pretrained weights**, and it reports the
   tail metric that the published work does not.

The honest conclusion is narrower than the headline metric alone would support, which is the
point of evaluating independently.